# Setup (Google Colab)

1. Upload `caption_data` to Google Drive at `MyDrive/DeepL-Final/Data/caption_data/` (folder must contain `captions.txt` and `Images/`).
2. Or upload `caption_data.zip` to `MyDrive/DeepL-Final/Data/` and run the next cell — it will unzip automatically.
3. Runtime → Change runtime type → **GPU** (recommended).

In [ ]:
from google.colab import drive
import zipfile
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/DeepL-Final')
DATA_DIR = PROJECT_ROOT / 'Data' / 'caption_data'
IMAGES_DIR = DATA_DIR / 'Images'
CAPTIONS_FILE = DATA_DIR / 'captions.txt'
MODEL_DIR = PROJECT_ROOT / 'models'
DATA_ZIP = PROJECT_ROOT / 'Data' / 'caption_data.zip'

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not CAPTIONS_FILE.exists() and DATA_ZIP.exists():
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(PROJECT_ROOT / 'Data')

assert CAPTIONS_FILE.exists(), f'Missing {CAPTIONS_FILE}. Upload caption_data to Drive first.'
assert IMAGES_DIR.exists(), f'Missing {IMAGES_DIR}.'

print('Data OK:', CAPTIONS_FILE)
print('Models will be saved to:', MODEL_DIR)

# Data Loading

In [ ]:
import json
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EMBED_DIM = 256
HIDDEN_DIM = 512
BATCH_SIZE = 64 if torch.cuda.is_available() else 32
NUM_EPOCHS = 12
LEARNING_RATE = 1e-3
MAX_SEQ_LEN = 40
MIN_WORD_FREQ = 5
NUM_WORKERS = 2

START_TOKEN = '<start>'
END_TOKEN = '<end>'
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'

In [ ]:
def tokenize_caption(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()


def build_vocabulary(captions, min_freq=MIN_WORD_FREQ):
    counter = Counter()
    for caption in captions:
        counter.update(tokenize_caption(caption))
    words = [word for word, freq in counter.items() if freq >= min_freq]
    words = sorted(words)
    special = [PAD_TOKEN, START_TOKEN, END_TOKEN, UNK_TOKEN]
    vocab = special + words
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word


def encode_caption(caption, word2idx, max_len=MAX_SEQ_LEN):
    tokens = [START_TOKEN] + tokenize_caption(caption) + [END_TOKEN]
    unk_idx = word2idx[UNK_TOKEN]
    pad_idx = word2idx[PAD_TOKEN]
    indices = [word2idx.get(token, unk_idx) for token in tokens]
    if len(indices) > max_len:
        indices = indices[: max_len - 1] + [word2idx[END_TOKEN]]
    length = len(indices)
    indices += [pad_idx] * (max_len - len(indices))
    return torch.tensor(indices, dtype=torch.long), length


caption_df = pd.read_csv(CAPTIONS_FILE)
image_names = sorted(caption_df['image'].unique())
random.shuffle(image_names)

n_total = len(image_names)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)

train_images = set(image_names[:n_train])
val_images = set(image_names[n_train : n_train + n_val])
test_images = set(image_names[n_train + n_val :])

train_df = caption_df[caption_df['image'].isin(train_images)].reset_index(drop=True)
val_df = caption_df[caption_df['image'].isin(val_images)].reset_index(drop=True)
test_df = caption_df[caption_df['image'].isin(test_images)].reset_index(drop=True)

word2idx, idx2word = build_vocabulary(train_df['caption'].tolist())
VOCAB_SIZE = len(word2idx)

print(f'Images: train={len(train_images)}, val={len(val_images)}, test={len(test_images)}')
print(f'Captions: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
print(f'Vocabulary size: {VOCAB_SIZE}')

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class CaptionDataset(Dataset):
    def __init__(self, dataframe, images_dir, word2idx, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.word2idx = word2idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.images_dir / row['image']
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        caption_tensor, caption_length = encode_caption(row['caption'], self.word2idx)
        return image, caption_tensor, caption_length


train_dataset = CaptionDataset(train_df, IMAGES_DIR, word2idx, image_transform)
val_dataset = CaptionDataset(val_df, IMAGES_DIR, word2idx, image_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)

sample_image, sample_caption, sample_length = train_dataset[0]
print(sample_image.shape, sample_caption.shape, sample_length)

# Model Definition

In [ ]:
class ImageCaptionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx):
        super().__init__()
        self.pad_idx = pad_idx
        self.hidden_dim = hidden_dim

        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        modules = list(resnet.children())[:-1]
        self.encoder = nn.Sequential(*modules)
        self.feature_dim = 512

        self.init_h = nn.Linear(self.feature_dim, hidden_dim)
        self.init_c = nn.Linear(self.feature_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def encode_image(self, images):
        features = self.encoder(images)
        features = features.view(features.size(0), -1)
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)
        return h0, c0

    def forward(self, images, captions):
        h0, c0 = self.encode_image(images)
        embeddings = self.dropout(self.embedding(captions))
        outputs, _ = self.lstm(embeddings, (h0, c0))
        logits = self.fc(self.dropout(outputs))
        return logits


pad_idx = word2idx[PAD_TOKEN]
model = ImageCaptionModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, pad_idx).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {total_params:,} total, {trainable_params:,} trainable')

# Training

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    total_tokens = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, captions, lengths in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            captions = captions.to(DEVICE, non_blocking=True)

            inputs = captions[:, :-1]
            targets = captions[:, 1:]

            logits = model(images, inputs)
            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1),
            )

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

            mask = targets != pad_idx
            total_loss += loss.item() * mask.sum().item()
            total_tokens += mask.sum().item()

    return total_loss / max(total_tokens, 1)


history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer)
    val_loss = run_epoch(model, val_loader)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
        f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f}'
    )

if best_state is not None:
    model.load_state_dict(best_state)

epochs = range(1, NUM_EPOCHS + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, history['train_loss'], marker='o', label='Train loss')
plt.plot(epochs, history['val_loss'], marker='o', label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Training progression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save

In [ ]:
checkpoint_path = MODEL_DIR / 'caption_model.pt'
vocab_path = MODEL_DIR / 'vocabulary.json'
config_path = MODEL_DIR / 'config.json'
history_path = MODEL_DIR / 'training_history.json'
splits_path = MODEL_DIR / 'data_splits.json'

torch.save(model.state_dict(), checkpoint_path)

with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(
        {'word2idx': word2idx, 'idx2word': {str(k): v for k, v in idx2word.items()}},
        f,
        indent=2,
    )

config = {
    'embed_dim': EMBED_DIM,
    'hidden_dim': HIDDEN_DIM,
    'max_seq_len': MAX_SEQ_LEN,
    'vocab_size': VOCAB_SIZE,
    'pad_idx': pad_idx,
    'start_token': START_TOKEN,
    'end_token': END_TOKEN,
    'pad_token': PAD_TOKEN,
    'unk_token': UNK_TOKEN,
    'image_size': 224,
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
}

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2)

with open(splits_path, 'w', encoding='utf-8') as f:
    json.dump(
        {
            'seed': SEED,
            'train_images': sorted(train_images),
            'val_images': sorted(val_images),
            'test_images': sorted(test_images),
        },
        f,
        indent=2,
    )

print(f'Saved model to {checkpoint_path}')
print(f'Saved vocabulary to {vocab_path}')
print(f'Saved config to {config_path}')